# 01 – Agents Overview & Base Contract

This notebook explores the **agent base classes**: `AgentRequest`, `AgentResult`, and `BaseAgent`.  
All five specialist agents inherit from these — understanding them is the foundation for everything else.

**No external credentials needed** — everything here is pure Python.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

## 1. AgentRequest — input contract

In [ ]:
from core.base_agent import AgentRequest, AgentResult

# Minimal request
req = AgentRequest(query="Why did retention drop last month?")
print("query      :", req.query)
print("thread_id  :", req.thread_id)
print("time_range :", req.time_range)
print("products   :", req.data_products)

In [ ]:
# Full request with all fields
req_full = AgentRequest(
    query="What is the DQ score for bookings?",
    thread_id="session-abc",
    user_id="alice@company.com",
    time_range="last_quarter",
    data_products=["bookings", "retention"],
    context={"source": "teams_bot"},
    intent="data_quality",
)
print(req_full)

## 2. AgentResult — output contract

In [ ]:
# Successful result
ok = AgentResult(
    success=True,
    data={"score": 94.2, "passed": 47, "failed": 3},
    message="DQ score retrieved for bookings",
    confidence=0.95,
    sources=["Collibra/bookings_fact"],
    execution_time_ms=42.3,
)
print("success    :", ok.success)
print("message    :", ok.message)
print("summary    :", ok.summary)  # alias for message
print("data       :", ok.data)
print("confidence :", ok.confidence)
print("sources    :", ok.sources)

In [ ]:
# Failure result (classmethod)
fail = AgentResult.failure("Databricks connection timeout", error="ConnectionError: timed out")
print("success :", fail.success)
print("error   :", fail.error)
print("message :", fail.message)

## 3. BaseAgent — health check interface

In [ ]:
from core.base_agent import BaseAgent

# Build a minimal concrete agent
class EchoAgent(BaseAgent):
    @property
    def name(self): return "echo_agent"
    
    def execute(self, request: AgentRequest) -> AgentResult:
        return AgentResult(
            success=True,
            message=f"Echo: {request.query}",
            confidence=1.0,
        )

agent = EchoAgent()
result = agent.execute(AgentRequest(query="hello world"))
print("message:", result.message)

In [ ]:
# Built-in health check
health = agent.health_check()
print(health)

## 4. All five agents — quick health check

In [ ]:
from agents.information_agent import InformationAgent
from agents.knowledge_agent import KnowledgeAgent
from agents.metadata_agent import MetadataAgent
from agents.capacity_agent import CapacityAgent
from agents.rule_agent import RuleAgent

agents = [
    InformationAgent(),
    KnowledgeAgent(),
    MetadataAgent(),
    CapacityAgent(),
    RuleAgent(),
]

for a in agents:
    h = a.health_check()
    status = '✅' if h.get('healthy') else '❌'
    print(f"{status} {h['agent']:25s}  latency={h.get('latency_ms', h.get('latency', 'N/A'))}ms")